# **Índice de adaptación de un jugador a un equipo (*suitability*)**

Este estudio calcula un **índice de adaptación** (*suitability*) que estima cómo de bien encajaría un jugador en un equipo concreto (en el TFM, el FC Barcelona). No mide solo "lo bueno que es", sino **cómo de compatible es** con el estilo, la plantilla, el contexto y el nivel del club de destino.

El resultado se entrega de dos formas: un **score de adaptación (%)** y una valoración en **estrellas (0.5 a 5)**.

In [1]:
import pandas as pd
import os
import numpy as np
from typing import Tuple
import utils.similarity_score as sim_score
from sklearn.metrics.pairwise import cosine_similarity

DATA_PATH = r"d:\data"
ACT_SEASON = "2526"
SILVER_DATA_PATH = os.path.join(DATA_PATH, "silver")

# Lectura de todos los dataframes (basicos)
MANAGER = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_1", "manager_clean_1.csv"), sep=";")
PLAYER = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_3", "player_clean_3_2.csv"), sep=";")
TEAM = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_2", "team_clean_2.csv"), sep=";")
MATCH = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_2", "match_clean_2.csv"), sep=";")

# Estadísticas
STATS_PLAYER = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_3", "player_stats_clean_3_2.csv"), sep=";")
STATS_TEAM = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_2", "team_stats_clean_2.csv"), sep=";")

# Mapas de pases y tiros
PASS_MAP = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_3", "pass_map_clean_3.csv"), sep=";")
SHOT_MAP = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_3", "shot_map_clean_3.csv"), sep=";")

# Dataframes con estadísticas agregadas de la temporada
PLAYER_AGG_SEASON = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_3", "agg_player.csv"), sep=";")
TEAM_AGG_SEASON = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_3", "agg_team.csv"), sep=";")

---

## **1. Preparación de los datos**

Se cargan y combinan muchas fuentes (jugadores, equipos, partidos, estadísticas, mapas de pase/tiro y agregados de temporada). En este bloque se preparan los ingredientes que luego alimentan las medidas de compatibilidad:

- **Matrices de similitud** entre jugadores (por grupo de posición) y entre equipos, reutilizando el estudio de *similarity score* (`utils/similarity_score`).
- **Formaciones y competiciones** en las que ha participado cada jugador y cada equipo durante la temporada.
- **Similitud jugador–equipo por estilo estadístico.** Se construyen percentiles de un conjunto de métricas por jugador y por equipo (equipos con ≥ 10 partidos, jugadores con ≥ 500 minutos) y se calcula su ***cosine similarity***, obteniendo cómo de parecido es el perfil estadístico de un jugador al de un equipo.


In [2]:
# OBTENCIÓN DE TODOS LOS DATOS
def obtain_all_data() -> dict:

    return_dict = {}

    # Obtención de la matrix de similitud de jugadores y de equipos
    corr_perc_goalkeepers, corr_perc_defenders, corr_perc_midfielders, corr_perc_attackers = sim_score.main_proc_data_similarity(match_info_df=MATCH.copy(), player_stats_df=STATS_PLAYER.copy(), player_info_df=PLAYER.copy())
    corr_perc_teams = sim_score.proc_teams_sim_matrix(matches_info_df=MATCH.copy(), team_season_agg_stats=TEAM_AGG_SEASON.copy())

    # Añadimos datos al diccionario
    return_dict["CorrelationGK"] = corr_perc_goalkeepers
    return_dict["CorrelationDF"] = corr_perc_defenders
    return_dict["CorrelationMD"] = corr_perc_midfielders
    return_dict["CorrelationFW"] = corr_perc_attackers
    return_dict["CorrelationTeam"] = corr_perc_teams

    # Obtención de las estadísticas de los partidos de la temporada
    matches_to_look = MATCH[MATCH["Season"].astype(str) == ACT_SEASON]["ID"].unique().tolist()
    player_stats = STATS_PLAYER[STATS_PLAYER["Match"].isin(matches_to_look)]
    team_stats = STATS_TEAM[STATS_TEAM["Match"].isin(matches_to_look)]

    # Añadimos formaciones
    team_match_formation = team_stats[["Match", "Team", "Formation"]]
    player_stats = player_stats.merge(team_match_formation, on=["Match", "Team"], how="inner")

    # Lista de formaciones posibles por equipo y por jugador
    player_formations_dict = (player_stats.groupby("Player")["Formation"].agg(lambda x: list(x.unique())).to_dict())
    team_formations_dict = (team_stats.groupby("Team")["Formation"].agg(lambda x: list(x.unique())).to_dict())

    # Añadimos competición a los datos por equipo y por jugador
    season_matches = MATCH[MATCH["Season"].astype(str) == ACT_SEASON]
    matches_tourn_dict = dict(zip(season_matches["ID"], season_matches["League"]))
    player_stats["League"] = player_stats["Match"].map(matches_tourn_dict)
    team_stats["League"] = team_stats["Match"].map(matches_tourn_dict)

    # Lista de competiciones en la que ha jugado un equipo y un jugador
    player_tournaments_dict = (player_stats.groupby("Player")["League"].agg(lambda x: list(x.unique())).to_dict())
    team_tournaments_dict = (team_stats.groupby("Team")["League"].agg(lambda x: list(x.unique())).to_dict())

    # Añadimos datos al diccionario
    return_dict["PlayerFormations"] = player_formations_dict
    return_dict["TeamFormations"] = team_formations_dict
    return_dict["PlayerTournaments"] = player_tournaments_dict
    return_dict["TeamTournaments"] = team_tournaments_dict

    min_matches = 10         # Partidos mínimos
    min_minutes = 500        # Minutos mínimos

    metrics = ["TouchesPer90", "PassesPer90", "AccuratePassesPer90", "PassAccuracy", "OppositionHalfPassesPer90", "AccurateOppositionHalfPassesPer90", "OppositionHalfPassAccuracy",
              "LongBallsPer90", "AccurateLongBallsPer90", "LongBallAccuracy", "ProgressiveFieldTilt", "PossessionLostPer90", "DispossessedPer90", "UnsuccessfulTouchesPer90",
              "KeyPassesPer90", "ExpectedAssistsPer90", "GoalAssistsPer90", "BigChancesCreatedPer90", "CrossesPer90", "AccurateCrossesPer90", "CrossAccuracy", "ContestsPer90", 
              "ContestsWonPer90", "ContestWinRate", "TotalShotsPer90", "ShotsOnTargetPer90", "ShotAccuracy", "ExpectedGoalsPer90", "ExpectedGoalsPerShot", "GoalsPer90", 
              "GoalConversion", "BigChancesMissedPer90", "OffsidesPer90", "TacklesPer90", "TacklesWonPer90", "TackleAccuracy", "InterceptionsPer90", "ClearancesPer90",
              "OutfielderBlocksPer90", "BallRecoveriesPer90", "DefensiveActionsPer90", "DuelsWonPer90", "DuelWinRate", "AerialWonPer90", "AerialWinRate", "ErrorsLeadToShotPer90",
              "FoulsPer90", "WasFouledPer90", "YellowCardsPer90", "SavesPer90", "SaveRate", "GoalsConcededPer90", "GoalsPreventedPer90", "HighClaimsPer90", "CrossesNotClaimedPer90", 
              "KeeperSweeperActionsPer90", "KeeperSweeperAccuracy", "PenaltySavesPer90", "PenaltySaveRate", "CleanSheets"]

    # Obtención del dataframe de porcentiles por equipo
    team_percentiles = TEAM_AGG_SEASON.copy()
    team_percentiles = team_percentiles[team_percentiles["Matches"] >= min_matches].set_index("Team")[metrics].fillna(0)
    team_percentiles = team_percentiles.rank(pct=True)

    # Obtención del dataframde porcentiles por jugador
    player_percentiles = PLAYER_AGG_SEASON.copy()
    player_percentiles = player_percentiles[player_percentiles["MinutesPlayed"] >= min_minutes].set_index("Player")[metrics].fillna(0)

    # Asegurarse de que las columnas están alineadas
    common_cols = player_percentiles.columns.intersection(team_percentiles.columns)
    X_players = player_percentiles[common_cols].fillna(0)
    X_teams = team_percentiles[common_cols].fillna(0)

    # Similaridad jugador-equipo
    similarity_matrix = cosine_similarity(X_players, X_teams)
    similarity_df = pd.DataFrame(similarity_matrix, index=player_percentiles.index, columns=team_percentiles.index)
    return_dict["SimilarityPlayerTeam"] = similarity_df

    return return_dict

In [3]:
dict_all_data = obtain_all_data()

---

## **2. Medidas de comparación jugador–equipo**

La función principal calcula, para un par (jugador, nuevo equipo), un conjunto de **medidas de compatibilidad**:

- **TeamsCorrelation** — similitud entre el equipo actual del jugador y el de destino (¿juegan parecido?).
- **SquadsCorrelation** — similitud media entre las plantillas de ambos equipos (excluyendo al propio jugador).
- **SamePosCorrelation** — similitud media del jugador con los jugadores del equipo de destino que ocupan **su misma posición** (¿se parece a quien ya está ahí?).
- **PassesCorrelation** — correlación entre el **mapa de calor de pases** del jugador y el de los jugadores de su posición en el equipo de destino.
- **FormationsMatch** — ¿comparten algún **grupo de formación** (A–E)? (binario).
- **TournamentsMatch** — ¿comparten algún **grupo de competición** (A–E) por nivel? (binario).
- **CountryMatch** — 1 si el país del jugador es el del club, 0.5 si hay compatriotas en la plantilla, 0 si no.
- **StatsCorrelation** — la *cosine similarity* jugador–equipo por estilo estadístico (§1).
- **Diferencias de contexto** — diferencia de edad media (`TimestampDiff`), de Elo (`EloRatingDiff`), de rating medio (`RatingDiff`) y de potencial medio (`PotentialDiff`) entre el jugador y el equipo de destino.

### **2.1. Similitud de mapas de pase**

Para `PassesCorrelation`, las posiciones de inicio de pase se discretizan en un **histograma 2D** (24×16 celdas sobre un campo 100×100) y se normaliza. La similitud entre el mapa del jugador ($H_a$) y el del equipo ($H_b$) es la **correlación de Pearson** entre ambos histogramas aplanados:

$$
r = \frac{\sum_i (a_i - \bar a)(b_i - \bar b)}{\sqrt{\sum_i (a_i - \bar a)^2}\,\sqrt{\sum_i (b_i - \bar b)^2}}
$$

Esto compara **dónde** toca/distribuye el balón el jugador frente a la zona típica de su posición en el equipo de destino.

In [4]:
# Obtención de la similitud entre dos plantillas
def get_squads_similarity(dict_all_data: dict, player_id: str, act_team_squad: pd.DataFrame, new_team_squad: pd.DataFrame, exclude_player: bool = True) -> float:

    # Obtención de similitud entre dos jugadores
    def get_players_similarity(corr_gk: pd.DataFrame, corr_df: pd.DataFrame, corr_mf: pd.DataFrame, corr_fw: pd.DataFrame, p1: str, p2: str) -> float:
        if p1 in corr_gk.columns and p2 in corr_gk.columns:
            return float(corr_gk[p1][p2])
        elif p1 in corr_df.columns and p2 in corr_df.columns:
            return float(corr_df[p1][p2])
        elif p1 in corr_mf.columns and p2 in corr_mf.columns:
            return float(corr_mf[p1][p2])
        elif p1 in corr_fw.columns and p2 in corr_fw.columns:
            return float(corr_fw[p1][p2])
        else:
            return None

    # Dataframes de correlación
    corr_gk = dict_all_data.get("CorrelationGK")
    corr_df = dict_all_data.get("CorrelationDF")
    corr_mf = dict_all_data.get("CorrelationMD")
    corr_fw = dict_all_data.get("CorrelationFW")

    # Obtenemos una lista con todas las similitudes
    all_sim_scores = []
    for act_team_player_id in act_team_squad["ID"].unique().tolist():
        if exclude_player:
            if act_team_player_id != player_id:
                for new_team_player_id in new_team_squad["ID"].unique().tolist():
                    similarity = get_players_similarity(corr_gk=corr_gk, corr_df=corr_df, corr_mf=corr_mf, corr_fw=corr_fw, p1=act_team_player_id, p2=new_team_player_id)
                    if similarity:
                        all_sim_scores.append(similarity)
        else:
            for new_team_player_id in new_team_squad["ID"].unique().tolist():
                similarity = get_players_similarity(corr_gk=corr_gk, corr_df=corr_df, corr_mf=corr_mf, corr_fw=corr_fw, p1=act_team_player_id, p2=new_team_player_id)
                if similarity:
                    all_sim_scores.append(similarity)

    # Media de similitud
    mean_squads_similarity = sum(all_sim_scores) / len(all_sim_scores)
    return mean_squads_similarity

# Obtener la similiud de mapa de pases
def get_passes_similarity(player_id: str, list_team_ids: list) -> float:

    def obtain_heatmap(list_pairs: list, normalize: bool = True) -> np.ndarray:
        x = np.array([p[0] for p in list_pairs])
        y = np.array([p[1] for p in list_pairs])
        heatmap, _, _ = np.histogram2d(x, y, bins=(24, 16), range=[[0, 100], [0, 100]])

        if normalize and heatmap.sum() > 0:
            heatmap = heatmap / heatmap.sum()

        return heatmap

    def correlate_heatmaps(heatmap_a: np.ndarray, heatmap_b: np.ndarray) -> float:
        if heatmap_a.sum() == 0 or heatmap_b.sum() == 0:
            return np.nan

        corr = np.corrcoef(heatmap_a.flatten(), heatmap_b.flatten())[0, 1]
        return corr

    # Mapa de pases
    pass_map_positions = PASS_MAP[["Player", "IniX", "IniY"]]

    # Obtenemos pases del jugador y pases de los jugadores de la misma posición del nuevo equipo
    player_pass_map = pass_map_positions[pass_map_positions["Player"] == player_id]
    team_pass_map = pass_map_positions[pass_map_positions["Player"].isin(list_team_ids)]

    # Lista de pares
    heatmap_player = obtain_heatmap(list_pairs=list(zip(player_pass_map["IniX"], player_pass_map["IniY"])))
    heatmap_team = obtain_heatmap(list_pairs=list(zip(team_pass_map["IniX"], team_pass_map["IniY"])))

    # Obtenemos correlación
    return float(correlate_heatmaps(heatmap_a=heatmap_player, heatmap_b=heatmap_team))

# Función principal para obtener las medidas para obtener la suitability de un jugador en un equipo
def get_player_team_similarity_measurements(dict_all_data: dict, team_id: str, player_id: str) -> dict:

    # Obtenemos equipo actual, y plantilla del equipo actual y nuevo equipo
    act_player_team = PLAYER[PLAYER["ID"] == player_id]["Team"].unique().tolist()[0]
    act_team_squad = PLAYER[PLAYER["Team"] == act_player_team]
    new_team_squad = PLAYER[PLAYER["Team"] == team_id]

    if team_id == act_player_team:
        return None

    # Correlación entre los dos equipos y plantillas (la correlación entre plantillas no tiene en cuenta el jugador)
    teams_correlation = float(dict_all_data.get("CorrelationTeam")[team_id][act_player_team])
    squads_correlation = get_squads_similarity(dict_all_data=dict_all_data, player_id=player_id, act_team_squad=act_team_squad, new_team_squad=new_team_squad)

    # Grupos de formaciones
    formation_group = {"4-2-3-1": "A", "4-1-4-1": "A", "4-3-2-1": "A", "4-4-1-1": "A", "4-5-1": "A", "4-4-2": "B", "4-2-2-2": "B", "4-1-3-2": "B", "4-3-1-2": "B", "4-3-3": "C", "3-4-3": "D", "3-2-4-1": "D", "3-3-3-1": "D",
                       "3-3-1-3": "D", "3-4-2-1": "D", "3-4-1-2": "D", "3-1-4-2": "D", "3-5-2": "E", "3-5-1-1": "E", "5-3-2": "E", "5-4-1": "E"}

    # Obtención de formaciones que ha jugado el jugador y el nuevo equipo
    player_posible_formations = set([formation_group[c] for c in dict_all_data["PlayerFormations"].get(player_id, [])])
    new_team_posible_formations = set([formation_group[c] for c in dict_all_data["TeamFormations"].get(team_id, [])])
    formation_match = int(bool(player_posible_formations & new_team_posible_formations))

    # Liga y grupo de liga para comparar
    competition_group = {"champions_league": "A", "premier_league": "A", "la_liga": "A", "bundesliga": "A", "serie_a": "A", "ligue_1": "A", "europa_league": "B", "liga_portugal": "B", "eredivise": "B", "super_lig": "B", "first_division_a": "B",
                        "championship": "B", "conference_league": "C", "copa_del_rey": "C", "la_liga_hypermotion": "C", "swiss_super_league": "C", "superligaen": "C", "saudi_pro_league": "C", "copa_libertadores": "D", "conmebol_sudamericana": "D",
                        "serie_a_brazil": "D", "liga_profesional": "D", "liga_mx": "D", "major_league_soccer": "D", "eliteserien": "E", "allvenskan": "E"}

    # Obtención de las competiciones que ha jugado el jugador y el nuevo equipo
    player_tournaments = set([competition_group[c] for c in dict_all_data["PlayerTournaments"].get(player_id, [])])
    new_team_tournaments = set([competition_group[c] for c in dict_all_data["TeamTournaments"].get(team_id, [])])
    tournaments_match = int(bool(player_tournaments & new_team_tournaments))

    # Booleano que mira si en el equipo hay un jugador del país del propio jugador
    player_country = PLAYER[PLAYER["ID"] == player_id]["Country"].unique().tolist()[0]
    new_team_countries = new_team_squad["Country"].unique().tolist()                        # Pondera 0.5 si es país que hay algun jugador
    new_team_country = TEAM[TEAM["ID"] == team_id]["Country"].unique().tolist()[0]          # Pondera 1 si es el país del equipo
    country_match = 1 if player_country == new_team_country else 0.5 if player_country in new_team_countries else 0

    # Lista de posiciones del jugador, y jugadores en sus mismas posiciones en el nuevo equipo
    player_positions = positions = [p for p in PLAYER.loc[PLAYER["ID"] == player_id, ["FirstPos", "SecondPos", "ThirdPos"]].values.flatten().tolist() if pd.notna(p)]
    same_position_new_team_players = new_team_squad[(new_team_squad["FirstPos"].isin(player_positions)) | (new_team_squad["SecondPos"].isin(player_positions)) | (new_team_squad["ThirdPos"].isin(player_positions))]
    same_position_mean_similarity = get_squads_similarity(dict_all_data=dict_all_data, player_id=player_id, act_team_squad=PLAYER.loc[PLAYER["ID"] == player_id], new_team_squad=same_position_new_team_players, exclude_player=False)

    # Fecha del jugador y media de jugadores del equipo
    act_player_timestamp = float(pd.to_datetime(PLAYER.loc[PLAYER["ID"] == player_id, "DateBirth"].iloc[0], format="%d/%m/%Y").timestamp())
    new_team_mean_timestamp = float(pd.to_datetime(PLAYER.loc[PLAYER["Team"] == team_id, "DateBirth"], format="%d/%m/%Y", errors="coerce").astype("int64").mean() / 1e9)

    # Elo de equipos
    act_team_elo = float(TEAM.loc[TEAM["ID"] == act_player_team, "EloRating"].iloc[0])
    new_team_elo = float(TEAM.loc[TEAM["ID"] == team_id, "EloRating"].iloc[0])

    # Rating del jugador y media del equipo
    player_rating = int(PLAYER.loc[PLAYER["ID"] == player_id, "Rating"].iloc[0]) 
    player_potential = int(PLAYER.loc[PLAYER["ID"] == player_id, "Potential"].iloc[0])
    new_team_rating = float(PLAYER.loc[PLAYER["Team"] == team_id, "Rating"].mean())
    new_team_potential = float(PLAYER.loc[PLAYER["Team"] == team_id, "Potential"].mean())

    # Obtención de la similitud con el mapa de pases
    pass_map_sim = get_passes_similarity(player_id=player_id, list_team_ids=same_position_new_team_players["ID"].unique().tolist())

    # Obtenemos la similaridad enttre el equipo y el jugador
    player_team_sim_dict = dict_all_data.get("SimilarityPlayerTeam")
    player_team_sim = float(player_team_sim_dict[team_id][player_id])

    # Diccionario de output
    output_dict = {"Player": player_id,
                  "Team": team_id,
                  "TeamsCorrelation": teams_correlation,
                  "SquadsCorrelation": squads_correlation, 
                  "FormationsMatch": formation_match,
                  "TournamentsMatch": tournaments_match,
                  "CountryMatch": country_match,
                  "SamePosCorrelation": same_position_mean_similarity,
                  "PassesCorrelation": pass_map_sim,
                  "TimestampDiff": new_team_mean_timestamp - act_player_timestamp,
                  "EloRatingDiff": new_team_elo - act_team_elo,
                  "RatingDiff": new_team_rating - player_rating,
                  "PotentialDiff": new_team_potential - player_potential,
                  "StatsCorrelation": player_team_sim}
    
    return output_dict

---

## **3. Selección de un subconjunto de jugadores**

Comparar todos los jugadores de la base de datos con el equipo sería ineficiente. Se filtra a los candidatos relevantes: jugadores con **≥ 10 partidos** en la temporada, con club, rating y potencial conocidos, ordenados por rating y potencial, quedándose con los **2000 mejores**.

In [5]:
# Obtención de los jugadores a consultar
def get_part_players(min_matches: int = 10) -> list:

    # Selección de los jugadores
    matches_to_look = MATCH[MATCH["Season"].astype(str)=="2526"]["ID"].unique().tolist()                                    # Jugadores que hayan jugado durante la temporada
    players_with_stats = STATS_PLAYER[STATS_PLAYER["Match"].isin(matches_to_look)]["Player"].unique().tolist()              # Jugadores con estadísticas
    players_agg_stats = PLAYER_AGG_SEASON[PLAYER_AGG_SEASON["Player"].isin(players_with_stats)]                             # Estadísticas agregadas de jugadores
    players_min_matches = players_agg_stats[players_agg_stats["Matches"] >= min_matches]["Player"].unique().tolist()        # Mínimo de partidos

    # Selección de jugadores final
    players_to_look_df = PLAYER[PLAYER["ID"].isin(players_min_matches)].dropna(subset="Team").dropna(subset="Rating").dropna(subset="Potential")
    players_to_look_df = players_to_look_df.sort_values(by=["Rating", "Potential"], ascending=False).head(2000)                 # Selección de los 2000 mejores jugadores del mundo
    players_to_look = players_to_look_df["ID"].unique().tolist()

    return players_to_look

In [6]:
# Obtenemos jugadores a mirar y diccionario con información
players_to_look = get_part_players()

---

## **4. Prueba para un equipo**

Se ejecuta el cálculo de todas las medidas para el equipo objetivo (FC Barcelona, `UXK2F`) sobre los candidatos, obteniendo un *dataframe* con una fila por jugador y una columna por medida.

In [8]:
team_id = "UXK2F"
total_len = len(players_to_look)
i = 1

# Concatenación de información
list_info = []
for player_id in players_to_look[:100]:             # Subset de los jugadores totales a mirar
    try:
        measures = get_player_team_similarity_measurements(dict_all_data=dict_all_data, team_id=team_id, player_id=player_id)
        list_info.append(measures)
        print(f"[{i}/{total_len}] Processed player {player_id} for team {team_id} - {round((i/total_len)* 100, 2)}%", end="\r", flush=True)
        i += 1
    except:
        continue

# Dataframe del equipo
df = pd.DataFrame(list_info)

---

## **5. Normalización y resultado final**

Las medidas tienen naturalezas distintas (porcentajes, binarios, diferencias), así que se llevan todas a una escala común $[0, 1]$ antes de combinarlas:

- **Correlaciones** (`TeamsCorrelation`, `SquadsCorrelation`, `SamePosCorrelation`, `PassesCorrelation`): se dividen por 100 y se acotan a $[0,1]$.
- **Binarios/tasas** (`FormationsMatch`, `TournamentsMatch`, `CountryMatch`, `StatsCorrelation`): ya están en $[0,1]$.
- **Diferencias** (`TimestampDiff`, `EloRatingDiff`, `RatingDiff`, `PotentialDiff`): se toma el valor absoluto y se normaliza de forma **invertida**, de modo que **una diferencia pequeña puntúa cerca de 1** (cuanto más parecido el contexto, mejor):

$$
\text{score}_c = 1 - \frac{|x| - \min|x|}{\max|x| - \min|x|}
$$

Los nulos sueltos se neutralizan con 0.5.

### **5.1. Score de adaptación**

El score final es una **media ponderada** de todas las medidas normalizadas:

$$
\text{MatchScore} = \frac{\sum_c w_c \cdot \text{score}_c}{\sum_c w_c}
$$

Los pesos $w_c$ reflejan la confianza en cada medida. Los más altos (1.0) son `TeamsCorrelation`, `SquadsCorrelation`, `EloRatingDiff`, `StatsCorrelation` y `RatingDiff`; los más bajos, factores secundarios como `CountryMatch` (0.2) o `TimestampDiff` (0.1).

### **5.2. Conversión a estrellas**

Para una lectura visual, el `MatchScore` se normaliza *min–max* y se mapea al rango **0.5 – 5**, redondeando al **0.5 más cercano**:

$$
\text{scaled} = \frac{s - \min s}{\max s - \min s}
\qquad
\text{Stars} = \frac{\mathrm{round}\big((\text{scaled}\cdot 9 + 1)\big)}{2}
$$


In [12]:
# Buscamos la matriz de correlación del equipo
correlation_df = pd.DataFrame([x for x in list_info if x is not None])

# Obtención de un score
def player_match_score(df: pd.DataFrame, weights=None) -> pd.DataFrame:

    df = df.copy()
    corr_cols = ["TeamsCorrelation","SquadsCorrelation","SamePosCorrelation","PassesCorrelation"]      # 0-100
    bin_cols  = ["FormationsMatch","TournamentsMatch","CountryMatch","StatsCorrelation"]               # ya 0-1
    diff_cols = ["TimestampDiff","EloRatingDiff","RatingDiff","PotentialDiff"]                         # cerca de 0 = mejor

    score = pd.DataFrame(index=df.index)

    for c in corr_cols:
        score[c] = (df[c] / 100).clip(0, 1)

    for c in bin_cols:
        score[c] = df[c].clip(0, 1)

    for c in diff_cols:
        a = df[c].abs()
        rng = a.max() - a.min()
        if pd.isna(rng) or rng == 0:      # columna constante o toda NaN
            score[c] = pd.NA              # no aporta info -> se neutraliza abajo
        else:
            score[c] = 1 - (a - a.min()) / rng   # diff pequeño -> cerca de 1

    # NaN sueltos (p.ej. PassesCorrelation, TimestampDiff) -> neutro
    score = score.fillna(0.5)

    # pesos (default: todos igual). Sube/baja segun lo que te fies de cada medida
    if weights is None:
        weights = {c: 1.0 for c in score.columns}
    w = pd.Series(weights).reindex(score.columns).fillna(0)

    df["MatchScore"] = (score * w).sum(axis=1) / w.sum()
    return df

# Normalización a estrellas
def to_stars_minmax(df, col="MatchScore") -> pd.DataFrame:
    df = df.copy()
    s = df[col]
    lo, hi = s.min(), s.max()
    scaled = (s - lo) / (hi - lo)          # 0-1
    stars = (scaled * 9 + 1) / 2           # lleva a 0.5 - 5
    df["Stars"] = (stars * 2).round() / 2  # redondea al 0.5 mas cercano
    return df

# Obtención del dataframe ponderado de jugadores con más adaptabilidad
weights = {"TeamsCorrelation": 1.0, "SquadsCorrelation": 1.0, "EloRatingDiff": 1.0,
        "StatsCorrelation": 1.0, "RatingDiff": 1.0, "SamePosCorrelation": 0.8,
        "PassesCorrelation": 0.8, "PotentialDiff": 0.7, "FormationsMatch": 0.3,
        "TournamentsMatch": 0.2, "CountryMatch": 0.2, "TimestampDiff": 0.1}
players_weighted = player_match_score(df=correlation_df, weights=weights)[["Player", "MatchScore"]]
players_stars = to_stars_minmax(df=players_weighted)        # Aplicamos estrellas
players_stars.columns = ["Player", "Score", "Stars"]        # Cambio de columnas

In [ ]:
players_stars["Player"] = players_stars["Player"].map(dict(zip(PLAYER["ID"], PLAYER["Name"])))
display(players_stars.sort_values(by=["Stars", "Score"], ascending=False).head(50))             # RECORDAR QUE ESTE ESTUDIO SOLO HA VISTO 100 DE LOS 2000 MEJORES JUGADORES - NO FIJARNOS EN LOS DATOS OBTENIDOS

,Player,Score,Stars
69,Michael Olise,0.746154,5.0
80,Kylian Mbappé,0.710769,4.5
65,Jurriën Timber,0.690386,4.0
77,William Saliba,0.689419,4.0
63,Erling Haaland,0.686197,4.0
53,Álvaro Carreras,0.685964,4.0
14,Dayot Upamecano,0.685303,4.0
31,Harry Kane,0.684813,4.0
33,Joshua Kimmich,0.669235,4.0
93,Ibrahima Konaté,0.668984,4.0
